In [1]:
# IMPORTAÇÕES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import glob

In [2]:
# CARREGAMENTO DOS ARQUIVOS

arquivos = glob.glob('*.csv')

print("Arquivos encontrados:")
print(arquivos)

# Escolhendo dois arquivos
df1 = pd.read_csv(arquivos[0])
df2 = pd.read_csv(arquivos[1])

# Identificando os arquivos
df1['categoria'] = arquivos[0]
df2['categoria'] = arquivos[1]

# Juntando os dados
df = pd.concat([df1, df2], ignore_index=True)

print("\nPrimeiras linhas:")
print(df.head())

Arquivos encontrados:
['us-shein-appliances-3987.csv', 'us-shein-automotive-4110.csv']

Primeiras linhas:
                              goods-title-link--jump  \
0  1pc Rechargeable Deep Tissue Muscle Handheld M...   
1                      1pc Portable Hanging Neck Fan   
2  1pc Pink Colored Curved Eyelash Curler False E...   
3  1 Mini Portable Handheld Fan With 2 Aa Batteri...   
4  Wit Water Flosser,Portable Oral Irrigator With...   

                         goods-title-link--jump href       rank-title  \
0  https://us.shein.com/1pc-Rechargeable-Deep-Tis...  #1 Best Sellers   
1  https://us.shein.com/1pc-Portable-Hanging-Neck...  #4 Best Sellers   
2  https://us.shein.com/1pc-Pink-Colored-Curved-E...              NaN   
3  https://us.shein.com/1-Mini-Portable-Handheld-...              NaN   
4  https://us.shein.com/Wit-Water-Flosser-Portabl...  #6 Best Sellers   

                        rank-sub   price discount  selling_proposition  \
0                  in Give Gifts   $2.03    

In [3]:
# DIAGNÓSTICO INICIAL (REGISTRAR PROBLEMAS ANTES DE CORRIGIR)

print("\nQuantidade de linhas e colunas:")
print(df.shape)

print("\nTipos de dados:")
print(df.dtypes)

print("\nValores ausentes:")
print(df.isnull().sum())

print("\nDuplicatas:")
print(df.duplicated().sum())

print("\nEstatísticas:")
print(df.describe(include='all'))


Quantidade de linhas e colunas:
(8095, 9)

Tipos de dados:
goods-title-link--jump         str
goods-title-link--jump href    str
rank-title                     str
rank-sub                       str
price                          str
discount                       str
selling_proposition            str
goods-title-link               str
categoria                      str
dtype: object

Valores ausentes:
goods-title-link--jump         8015
goods-title-link--jump href    8015
rank-title                     7536
rank-sub                       7536
price                             0
discount                       4263
selling_proposition            3860
goods-title-link                 80
categoria                         0
dtype: int64

Duplicatas:
388

Estatísticas:
                                   goods-title-link--jump  \
count                                                  80   
unique                                                 80   
top     1pc Rechargeable Deep Tissue Mus

In [4]:
# FUNÇÕES DO PIPELINE

def corrigir_tipos(df):

    df = df.copy()

    # Convertendo preços
    if 'price' in df.columns:
        df['price'] = df['price'].astype(str)
        df['price'] = df['price'].str.replace('$', '', regex=False)
        df['price'] = df['price'].str.replace(',', '', regex=False)
        df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # Convertendo descontos
    if 'discount' in df.columns:
        df['discount'] = df['discount'].astype(str)
        df['discount'] = df['discount'].str.replace('%', '', regex=False)
        df['discount'] = pd.to_numeric(df['discount'], errors='coerce')

    return df


def tratar_ausentes(df):

    df = df.copy()

    # Trocando marcadores por NaN
    df = df.replace(['', '?', 'N/A', 'null'], np.nan)

    # Removendo produtos sem preço
    if 'price' in df.columns:
        df = df.dropna(subset=['price'])

    return df


def tratar_duplicatas(df):

    df = df.copy()

    # Removendo linhas iguais
    df = df.drop_duplicates()

    return df


def padronizar_textos(df):

    df = df.copy()

    # Pegando as colunas de texto
    textos = df.select_dtypes(include='object').columns

    # Removendo espaços extras
    for coluna in textos:
        df[coluna] = df[coluna].str.strip()

    return df


def tratar_outliers(df):

    df = df.copy()

    # Removendo preços inválidos
    df = df[df['price'] > 0].copy()

    # Calculando os quartis
    q1 = df['price'].quantile(0.25)
    q3 = df['price'].quantile(0.75)

    iqr = q3 - q1

    minimo = q1 - 1.5 * iqr
    maximo = q3 + 1.5 * iqr

    # Limitando preços muito altos
    df['price'] = df['price'].clip(minimo, maximo)

    return df


def criar_features(df):

    df = df.copy()

    # Criando uma coluna de faixa de preço
    df['faixa_preco'] = pd.cut(
        df['price'],
        bins=[0, 10, 30, np.inf],
        labels=['Baixo', 'Médio', 'Alto']
    )

    return df


In [5]:
# EXECUÇÃO DO PIPELINE

# Criando uma cópia do original
df_limpo = df.copy()

# Aplicando as funções
df_limpo = padronizar_textos(df_limpo)
df_limpo = corrigir_tipos(df_limpo)
df_limpo = tratar_ausentes(df_limpo)
df_limpo = tratar_duplicatas(df_limpo)
df_limpo = tratar_outliers(df_limpo)
df_limpo = criar_features(df_limpo)

print("\nDados limpos:")
print(df_limpo.head())



Dados limpos:
                              goods-title-link--jump  \
0  1pc Rechargeable Deep Tissue Muscle Handheld M...   
1                      1pc Portable Hanging Neck Fan   
2  1pc Pink Colored Curved Eyelash Curler False E...   
3  1 Mini Portable Handheld Fan With 2 Aa Batteri...   
4  Wit Water Flosser,Portable Oral Irrigator With...   

                         goods-title-link--jump href       rank-title  \
0  https://us.shein.com/1pc-Rechargeable-Deep-Tis...  #1 Best Sellers   
1  https://us.shein.com/1pc-Portable-Hanging-Neck...  #4 Best Sellers   
2  https://us.shein.com/1pc-Pink-Colored-Curved-E...              NaN   
3  https://us.shein.com/1-Mini-Portable-Handheld-...              NaN   
4  https://us.shein.com/Wit-Water-Flosser-Portabl...  #6 Best Sellers   

                        rank-sub  price  discount  selling_proposition  \
0                  in Give Gifts   2.03     -22.0                  NaN   
1  in Top rated in Portable Fans   6.48     -20.0            

C:\Users\NONATO\AppData\Local\Temp\ipykernel_19424\1339063298.py:52: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  textos = df.select_dtypes(include='object').columns


In [6]:
# TABELA ANTES x DEPOIS

# Quantidade de linhas
linhas_antes = len(df)
linhas_depois = len(df_limpo)

# Porcentagem de valores ausentes
ausentes_antes = df.isnull().sum().sum() / df.size * 100
ausentes_depois = df_limpo.isnull().sum().sum() / df_limpo.size * 100

# Quantidade de duplicatas
duplicatas_antes = df.duplicated().sum()
duplicatas_depois = df_limpo.duplicated().sum()

# Criando a tabela
comparacao = pd.DataFrame({
    'Indicador': ['Linhas', 'Ausentes (%)', 'Duplicatas'],
    'Antes': [linhas_antes, round(ausentes_antes, 2), duplicatas_antes],
    'Depois': [linhas_depois, round(ausentes_depois, 2), duplicatas_depois]
})

print("\nComparação antes e depois:")
print(comparacao)

# Tipos das colunas
print("\nTipos antes:")
print(df.dtypes)

print("\nTipos depois:")
print(df_limpo.dtypes)




Comparação antes e depois:
      Indicador    Antes   Depois
0        Linhas  8095.00  7707.00
1  Ausentes (%)    53.95    48.48
2    Duplicatas   388.00     5.00

Tipos antes:
goods-title-link--jump         str
goods-title-link--jump href    str
rank-title                     str
rank-sub                       str
price                          str
discount                       str
selling_proposition            str
goods-title-link               str
categoria                      str
dtype: object

Tipos depois:
goods-title-link--jump              str
goods-title-link--jump href         str
rank-title                          str
rank-sub                            str
price                           float64
discount                        float64
selling_proposition                 str
goods-title-link                    str
categoria                           str
faixa_preco                    category
dtype: object


In [7]:
# TABELA ANTES x DEPOIS

# Quantidade de linhas
linhas_antes = len(df)
linhas_depois = len(df_limpo)

# Porcentagem de valores ausentes
ausentes_antes = df.isnull().sum().sum() / df.size * 100
ausentes_depois = df_limpo.isnull().sum().sum() / df_limpo.size * 100

# Quantidade de duplicatas
duplicatas_antes = df.duplicated().sum()
duplicatas_depois = df_limpo.duplicated().sum()

# Criando a tabela
comparacao = pd.DataFrame({
    'Indicador': ['Linhas', 'Ausentes (%)', 'Duplicatas'],
    'Antes': [linhas_antes, round(ausentes_antes, 2), duplicatas_antes],
    'Depois': [linhas_depois, round(ausentes_depois, 2), duplicatas_depois]
})

print("\nComparação antes e depois:")
print(comparacao)

# Tipos das colunas
print("\nTipos antes:")
print(df.dtypes)

print("\nTipos depois:")
print(df_limpo.dtypes)




Comparação antes e depois:
      Indicador    Antes   Depois
0        Linhas  8095.00  7707.00
1  Ausentes (%)    53.95    48.48
2    Duplicatas   388.00     5.00

Tipos antes:
goods-title-link--jump         str
goods-title-link--jump href    str
rank-title                     str
rank-sub                       str
price                          str
discount                       str
selling_proposition            str
goods-title-link               str
categoria                      str
dtype: object

Tipos depois:
goods-title-link--jump              str
goods-title-link--jump href         str
rank-title                          str
rank-sub                            str
price                           float64
discount                        float64
selling_proposition                 str
goods-title-link                    str
categoria                           str
faixa_preco                    category
dtype: object


In [ ]:
# RELATÓRIO E RESPOSTAS ÀS PERGUNTAS DE REFLEXÃO

# 1. Qual foi a pergunta de negócio?
# Quais categorias possuem os maiores preços
# e descontos médios?

# 2. Quais problemas foram analisados?
# Valores ausentes, duplicatas, textos,
# tipos de dados e outliers.

# 3. Como os dados foram tratados?
# Os preços e descontos foram convertidos.
# Os valores ausentes nos preços foram removidos.
# As duplicatas foram excluídas e os textos padronizados.

# 4. Qual coluna foi criada?
# Foi criada a coluna faixa_preco.

# 5. Por que utilizar funções?
# Para organizar o código e poder utilizar
# as mesmas funções em outros arquivos.

# 6. Como foi feita a comparação?
# Foram comparadas linhas, valores ausentes,
# duplicatas, tipos de dados e gráficos.
